# Algorithm 3 — the $K$ sweep at fixed $\lambda$

**Question.** At one fixed medium $\lambda$, does raising the lookahead depth $K$ change the
within-run $E_{q}[f]$ curve — specifically in the steps **before** the cloud degenerates?

This is a narrowing of "does $E_{q}[f]$ rise with $\lambda$". At $M = 8$ the cloud collapses to a
single root ancestor at $m = 2$ by step 2 of 16, so a $\lambda$ sweep measures degeneracy rather than
the estimator. Degeneracy at strong tilt is accepted as inherent to the method (systematic resampling
destroys lineages and nothing regenerates them) and is not what this notebook is about.

**Hypothesis.** The decrease is finite-$K$ estimator noise in the twist, not diversity loss.

## Design

Fixed: $M = 8$, seed 101, $\lambda = 1.25 \lambda_{s} = 175.1$, `ess_threshold=0.5`,
`guid_window=(0.0, 1.0)`, $\eta = 1.5$, $N = 16$, branch 1 (`use_full_normalized_v=False`).
**Swept:** $K \in \{4, 8, 16, 32\}$, one job per $K$, all submitted concurrently.

`K` and `M_TILT` come from the environment, so one notebook and one `.slurm` serve every cell.

**Base process frozen** — `n_steps=16`, `stoch_window=(0.1,1.0)`, `LinearSchedule`, `ts=None`,
prompt, guidance. `refs_R64_seed7.pt` and $\lambda_{s} = 140.093$ are copied verbatim from
`../flowmap_smc_flux/` and asserted against `BASE_PROCESS` on load, so no reference rebuild happens
and no $\lambda_{s}$ is re-measured.

**Sampler frozen.** No replay lookahead, no CRN. The only additions are recording flags
(`record_r_k`, `project_endpoint`), which consume no randomness and leave the trajectory
bit-identical — asserted by `test_recording_flags_leave_the_run_bit_identical`.

## Why $\lambda = 175$

From a3_2 (`ess_threshold=0.5`, $K=4$, $M=8$):

| $m$ | $\lambda$ | resamples | collapses to 1 lineage | terminal $f$ |
|---|---|---|---|---|
| 1 | 140.1 | **1** / 16 | never (holds 3) | 1.0087 |
| 2 | 280.2 | 8 / 16 | **step 2** | 0.9996 |
| 3 | 420.3 | 8 / 16 | step 7 | 0.9980 |

$m = 1$ is too weak — the mechanism fires once in 16 steps. $m = 2$ collapses at step 2, leaving no
pre-degeneration window. $\lambda = 175$ ($m = 1.25$) is the conservative choice inside that gap.
The collapse step is seed-noisy ($m=3$ collapsed *later* than $m=2$), so $\lambda$ is **not** tuned
finely against it — the $K=4$ job acts as a live scout and $\lambda$ is revised for all cells if its
first ~5 steps say the window is wrong.

## No IS baseline

The resampling-off IS numbers are **not** a comparand here. With resampling off the potential
telescopes exactly to $\lambda f(x_{1})$, so IS is a *different estimator*, not a better-behaved
version of this one. The legitimate references are in-job $E_{p}[f]$, the $\lambda = 0$ control curve,
and $K$ against $K$. In-job matters: a GPU change once shifted $f$ by 16% of $\operatorname{std}_{p}(f)$
(job 697271), the same order as the effects here — hence `--constraint=l40s`, one model, never a
disjunction.

## Reading caveat

At small $t$, $\operatorname{map}(x_{t}, t, 1)$ is a poor endpoint estimate, so early-step $f$ is not
the $f$ of a real sample. In a3_2 the candidate mean climbs 0.918 → 1.02 over the first ~8 steps and
most of that is the estimate sharpening, **not** tilt. Read the late plateau and the give-back
(peak − terminal), not the early rise, and never read anything right of a curve's collapse step.

In [1]:
import json
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import torch

T_START = time.time()

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(REPO, "creativity_measure")):
    REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO not in sys.path:
    sys.path.insert(0, REPO)
RUN_DIR = os.path.join(REPO, "notebooks", "flowmap_smc_k_sweep")
os.makedirs(RUN_DIR, exist_ok=True)

# --- what to run ----------------------------------------------------------------------------------
# Phases are cumulative and cache to disk. This directory ships with refs_R64_seed7.pt and
# lam_s_R64_seed7.json copied verbatim from ../flowmap_smc_flux, so "refs" is a no-op load and every
# job goes straight to "run". The gate is still worth one job before the sweep.
STOP_AFTER = os.environ.get("STOP_AFTER", "run")          # health | refs | run
assert STOP_AFTER in ("health", "refs", "run"), STOP_AFTER
DO_REFS = STOP_AFTER in ("refs", "run")
DO_RUN = STOP_AFTER == "run"
# One tag per cell of the sweep, so concurrent jobs never overwrite each other's outputs. Output
# names also carry M/K/m/seed, but the tag is what separates a re-run at a revised lambda.
RUN_TAG = os.environ.get("RUN_TAG", "k_sweep")

# --- wall clock -----------------------------------------------------------------------------------
# The SMC is one long library call, so it cannot be interrupted from outside; the `on_step` callback
# checks DEADLINE after every step instead. RESERVE_H is held back for decoding and rendering.
# Under Slurm, SLURM_JOB_END_TIME is the job's real end, so the deadline tracks --time automatically.
WALL_BUDGET_H = float(os.environ.get("WALL_BUDGET_H", 6.0))   # keep in sync with --time in the .slurm
RESERVE_H = 0.75                                              # VAE decoding, rendering, writing out
try:
    _end = float(os.environ["SLURM_JOB_END_TIME"])
except (KeyError, ValueError):
    _end = None
if _end and _end > T_START:
    DEADLINE, WALL_BUDGET_H, _src = _end - RESERVE_H * 3600, (_end - T_START) / 3600, "SLURM"
else:
    DEADLINE, _src = T_START + (WALL_BUDGET_H - RESERVE_H) * 3600, "constant"

from creativity_measure import set_default_dtype

set_default_dtype(torch.float32)          # FLUX runs bf16; everything the SMC touches is fp32
torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16                    # transformer compute dtype; particles stay fp32
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"

# --- progress log ----------------------------------------------------------------------------------
# `nbconvert --execute` captures cell stdout into the OUTPUT notebook, which is only written when the
# job ends -- so the Slurm .out log shows nothing for the whole run and there is no way to watch a
# long job from outside. `note()` mirrors the milestones that matter to a plain text file that can be
# tailed live. Line-buffered and flushed per call, so a killed job still leaves everything up to the
# last step on disk. THE WATCHDOGS TAIL THIS FILE -- do not change its name without updating them.
PROGRESS_PATH = os.path.join(RUN_DIR, f"progress_{RUN_TAG}_{STOP_AFTER}.log")


# --- peak host RSS, stamped on every progress line -------------------------------------------------
# WHY. `--mem` was inherited superstition until it was measured. On 2026-08-26 it cost ~2 h of queue:
# n-601 had 2 FREE A6000s but only 11.8 GB of free RAM, so a 64 GB request could not land there.
# Slurm's accounting here does NOT collect MaxRSS (`sacct` returns it empty), so the number cannot be
# recovered after the fact -- it has to be taken from inside the process.
#
# ru_maxrss is a HIGH-WATER MARK, so stamping it on every line also shows WHERE the peak happens
# (model load vs reference bank vs the run), which is what decides whether --mem can drop.
#
# CAVEAT: Slurm's cgroup limit counts page cache for mmap'd files, which ru_maxrss under-reports. The
# 32 G of safetensors are read by mmap. So this is a LOWER bound on what Slurm charges -- set --mem
# with real headroom above the peak seen here, never equal to it.
import resource


def _peak_gb() -> float:
    """Peak resident set size of this process, in GB (ru_maxrss is KB on Linux)."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20


def note(msg: str) -> None:
    """Print AND append to PROGRESS_PATH, with a wall-clock offset and peak host RSS."""
    line = f"[{(time.time() - T_START) / 60:7.1f} min |{_peak_gb():5.1f}G] {msg}"
    print(line, flush=True)
    with open(PROGRESS_PATH, "a") as fh:
        fh.write(line + "\n")
        fh.flush()


# ADVISORY ONLY -- never fatal. This used to be a bare `auth_check(...)` and it killed two jobs in a
# row: 781449 on a 401 GatedRepoError ($HOME/.hf_token is stale; the only credential that ever worked
# was an HF_TOKEN exported in an interactive shell, which a submitted script does not inherit), then
# 781469 on OfflineModeIsEnabled once the slurm script set HF_HUB_OFFLINE=1 to dodge the first. The
# hub is not needed at all: the slurm script stages the full snapshot to node-local disk and
# `from_pretrained` on a directory holding model_index.json never resolves a repo id. The REAL guard
# is the FLUX_LOCAL_PATH / model_index.json assert below, which fails in seconds and names the cause.
try:
    from huggingface_hub import auth_check

    auth_check("black-forest-labs/FLUX.1-dev")
    HUB_ACCESS = "OK"
except Exception as exc:
    HUB_ACCESS = f"UNAVAILABLE ({type(exc).__name__})"
note(f"START phase={STOP_AFTER} tag={RUN_TAG} job={os.environ.get('SLURM_JOB_ID', 'manual')} "
     f"gpu={GPU_NAME}")
print(f"device {DEVICE} ({DTYPE})   budget {WALL_BUDGET_H:.2f} h via {_src}")
print(f"deadline {time.strftime('%H:%M:%S', time.localtime(DEADLINE))}")
print(f"STOP_AFTER={STOP_AFTER!r}  run tag {RUN_TAG!r}   FLUX.1-dev hub access: {HUB_ACCESS}")

[    0.0 min |  0.5G] START phase=run tag=k32 job=869715 gpu=NVIDIA L40S
device cuda (torch.bfloat16)   budget 21.73 h via SLURM
deadline 17:28:08
STOP_AFTER='run'  run tag 'k32'   FLUX.1-dev hub access: UNAVAILABLE (OfflineModeIsEnabled)


In [2]:
from diffusers import FluxPipeline

MODEL_ID = "black-forest-labs/FLUX.1-dev"
# LOAD PATH vs IDENTITY. MODEL_ID stays the canonical repo id because it is recorded in BASE_PROCESS
# and asserted against the persisted reference file -- swapping a local path in there would fail that
# check for no reason. MODEL_PATH is only where the weights are read from: the slurm script stages the
# 32 G snapshot to node-local disk and exports FLUX_LOCAL_PATH, because the NFS mount serves it at
# ~2 MB/s (a MEASURED 257 min cold, job 755539). `from_pretrained` on a directory holding
# model_index.json never resolves a repo id, which is also what makes HF_HUB_OFFLINE=1 safe.
MODEL_PATH = os.environ.get("FLUX_LOCAL_PATH") or MODEL_ID
if MODEL_PATH != MODEL_ID:
    assert os.path.isfile(os.path.join(MODEL_PATH, "model_index.json")), (
        f"FLUX_LOCAL_PATH={MODEL_PATH} has no model_index.json -- the slurm staging step did not "
        f"finish; loading would silently fall back to the hub")
    print(f"loading FLUX from node-local stage: {MODEL_PATH}")
else:
    print("FLUX_LOCAL_PATH unset -- falling back to the HF cache over NFS (~257 min cold)")
PROMPT = "A dog"

# GUIDANCE = 1.0 for the flow map (the pCN sweeps used 1.5 / 3.5 on base FLUX). Together with PROMPT
# and the checkpoint this IS p, so it is fixed here and never exposed to the library.
GUIDANCE = 1.0

IMG = 512
C, H, W = 16, IMG // 8, IMG // 8       # VAE downsamples 8x; FLUX latents have 16 channels
LATENT_DIM = C * H * W                 # 65536 -- the geometry the FLUX reward machinery is tuned for

pipe = FluxPipeline.from_pretrained(MODEL_PATH, torch_dtype=DTYPE).to(DEVICE)
with torch.no_grad():
    prompt_embeds, pooled_prompt_embeds, text_ids = pipe.encode_prompt(
        prompt=PROMPT, prompt_2=None, device=DEVICE, max_sequence_length=512
    )
# Free both text encoders (~9.5 GB, T5 dominates): the embeddings above are all we need.
pipe.text_encoder = pipe.text_encoder_2 = pipe.tokenizer = pipe.tokenizer_2 = None
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
transformer, vae = pipe.transformer.eval(), pipe.vae.eval()
VAE_SF, VAE_SHIFT = vae.config.scaling_factor, vae.config.shift_factor
print(f"loaded {MODEL_ID}  prompt={PROMPT!r}  guidance={GUIDANCE}  "
      f"guidance_embeds={transformer.config.guidance_embeds}")
print(f"latents ({C}, {H}, {W}) = d {LATENT_DIM}  vae scaling={VAE_SF} shift={VAE_SHIFT}")


def decode(flat: torch.Tensor, chunk: int = 2) -> torch.Tensor:
    """Flat FLUX latents (B, d) -> images (B, 3, 512, 512) in [0, 1].

    Chunked because the VAE decoder, not the transformer, is the memory spike.
    """
    outs = []
    for i in range(0, flat.shape[0], chunk):
        lat = flat[i : i + chunk].reshape(-1, C, H, W).to(DEVICE, DTYPE) / VAE_SF + VAE_SHIFT
        with torch.no_grad():
            img = vae.decode(lat).sample
        outs.append(pipe.image_processor.postprocess(img.float(), output_type="pt").cpu())
    return torch.cat(outs)

loading FLUX from node-local stage: /tmp/arielzoizner/flux-snap-3de623fc
loaded black-forest-labs/FLUX.1-dev  prompt='A dog'  guidance=1.0  guidance_embeds=True
latents (16, 64, 64) = d 65536  vae scaling=0.3611 shift=0.1159


In [3]:
from creativity_measure import LinearSchedule, ddpm_step, flow_map_step
from creativity_measure.distances.edm_adapter import chunked_denoiser, edm_score_fn
from creativity_measure.generators.flux_flowmap import (
    FluxFlowMap, flow_map_denoiser, load_flow_map_weights,
)

# --- denoiser batch cap ---------------------------------------------------------------------------
# MEASURED, job 695266 on this exact card/model/resolution (probe_denoiser_batch.py):
#     rows      24     48     96    128    192
#     s/row  0.259  0.251  0.255  0.263    OOM
# s/row is FLAT across the range, so a narrow cap costs nothing in throughput (same number of forward
# passes, same cost each) and leaves headroom for a multi-hour run. R=64 with NUM_EPS=3 needs 192 rows
# for the reference-bank build, which is exactly the width that OOMs on an A6000.
# FIXED for the whole run, so f stays a deterministic function of x (CLAUDE.md invariant 1).
MAX_DENOISER_ROWS = 24
SIGMA_MIN, SIGMA_MAX = 2e-3, 80.0

SCHEDULE = LinearSchedule()               # alpha_t = t, sigma_t = 1 - t; FLUX is rectified flow

# --- load the flow-map weights --------------------------------------------------------------------
# ORDER MATTERS: load_flow_map_weights wraps time_text_embed in a DualTimeEmbedder FIRST, or the
# second-embedder keys have nowhere to land. It asserts every tensor is consumed and that the loaded
# model differs from the unmodified one on a fixed input -- without that check a silently no-op load
# means sampling plain FLUX.1-dev while believing it is the flow map, and nothing downstream catches it.
load_flow_map_weights(transformer)

# max_rows caps the transformer's batch the same way MAX_DENOISER_ROWS caps the score path: the
# lookahead calls map() on M*K = 32 rows and reference generation on R = 64, against a model already
# holding ~24 GB. Fixed for the run -- chunking agrees only to float epsilon, so a varying cap would
# make f non-deterministic in x.
flow_map = FluxFlowMap(
    transformer, prompt_embeds, pooled_prompt_embeds, text_ids,
    guidance=GUIDANCE, img_shape=(C, H, W), dtype=DTYPE, max_rows=MAX_DENOISER_ROWS,
)

# ONE wrapper feeding BOTH the reward's distance and the sampler's S_k: the score model that defines
# f's geometry and the one in the score correction must be the same object on the same code path.
# `denoise`, never `map(., t, 1)` -- only a conditional mean backs a marginal score.
denoiser_capped = chunked_denoiser(flow_map_denoiser(flow_map, SCHEDULE), MAX_DENOISER_ROWS)
score_fn = edm_score_fn(denoiser_capped, (C, H, W))
note(f"flow map weights loaded; score ready (denoiser capped at {MAX_DENOISER_ROWS} rows)")


# --- the base process p ----------------------------------------------------------------------------
# ts, schedule, stoch_window and the two TransitionSteps ARE p. guid_window is deliberately NOT in this
# dict: it only decides where the reward runs, so it can be swept freely against one fixed p and one
# reference set. Changing anything below invalidates the references; changing guid_window does not.
N_STEPS = 16
BASE_PROCESS = {
    "schedule": SCHEDULE.name,
    "n_steps": N_STEPS,
    "ts": None,                                # None => linspace(0, 1, N_STEPS + 1)
    "stoch_window": (0.1, 1.0),
    "inside_step": ddpm_step.__name__,
    "outside_step": flow_map_step.__name__,
    "img_shape": [C, H, W],
    "model_id": MODEL_ID,
    "prompt": PROMPT,
    "guidance": GUIDANCE,
    "max_denoiser_rows": MAX_DENOISER_ROWS,
    "max_flow_map_rows": MAX_DENOISER_ROWS,
}
BASE_KW = dict(
    flow_map=flow_map, score_fn=score_fn, schedule=SCHEDULE, n_steps=N_STEPS,
    stoch_window=BASE_PROCESS["stoch_window"], inside_step=ddpm_step, outside_step=flow_map_step,
)
print("base process p:", json.dumps({k: v for k, v in BASE_PROCESS.items()
                                     if k not in ("model_id",)}, default=str))

flow map loaded: 692 tensors, 346 LoRA sites, max|delta| on a fixed input = 1.410e+00
[    2.0 min | 10.2G] flow map weights loaded; score ready (denoiser capped at 24 rows)
base process p: {"schedule": "linear", "n_steps": 16, "ts": null, "stoch_window": [0.1, 1.0], "inside_step": "ddpm_step", "outside_step": "flow_map_step", "img_shape": [16, 64, 64], "prompt": "A dog", "guidance": 1.0, "max_denoiser_rows": 24, "max_flow_map_rows": 24}


In [4]:
from creativity_measure import LpDistance, Reward, flowmap_smc_sample

# --- PHASE 1 GATE: is p itself healthy? ------------------------------------------------------------
# If p looks degraded, stop before building 3 h on top of it.

S_ROW = 0.206            # s per row, MEASURED on an L40S by job 700483's reference-bank build
                         # (5568 rows in 19.1 min) -- the EXACT reward code path the run spends
                         # ~97% of its time in, so it is the right number to budget with. The
                         # 8-row flow-map path measures slower (0.231) because per-call overhead
                         # is amortised over less work; the A6000 figure was 0.251.
S_BASE_FLUX = 1.0150     # latent std of base FLUX.1-dev at this resolution (job 695271)


def _dummy_reward(n: int = 2) -> Reward:
    """A reward that costs nothing, for drawing from p itself.

    `flowmap_smc_sample` needs a `Reward` to pin d/device/dtype, but at lam = 0 its *value* cannot
    affect the trajectory (V = 0 everywhere, and systematic resampling on uniform weights is the
    identity). An L2 reward against dummy references is pure tensor arithmetic -- no network. Paired
    with `guid_window=(1.0, 1.0)` this skips every lookahead, so an untilted draw costs exactly one
    flow-map row per particle per step and NOT the M(2K+2) a guided step would.
    """
    return Reward(distance=LpDistance(p=2.0),
                  x_refs=torch.zeros(n, LATENT_DIM, device=DEVICE, dtype=torch.float32))


def draw_from_p(n: int, seed: int, *, verbose: bool = False) -> torch.Tensor:
    """n untilted samples from p, through the SAME stepping code path the tilted run uses.

    This is what makes the references valid: they are drawn by the base process the run will sample,
    not by some other integrator.
    """
    res = flowmap_smc_sample(_dummy_reward(), 0.0, n, **BASE_KW,
                             guid_window=(1.0, 1.0), mc_samples=1, seed=seed, verbose=verbose)
    return res.X.detach()


t0_health = t0 = time.time()
health = draw_from_p(8, seed=1, verbose=True)
S_health = float(health.std())
print(f"\nuntilted draw: {tuple(health.shape)} in {time.time() - t0:.0f}s   "
      f"latent std = {S_health:.4f}  (base FLUX: {S_BASE_FLUX:.4f})")
print(f"  implied {(time.time() - t0) / (8 * N_STEPS):.3f} s/row vs {S_ROW} measured")

# --- gate 1: p is not degraded --------------------------------------------------------------------
assert 0.6 * S_BASE_FLUX < S_health < 1.6 * S_BASE_FLUX, (
    f"latent std {S_health:.4f} is far from base FLUX's {S_BASE_FLUX:.4f}: p looks degraded, and the "
    f"reward/lambda calibration built on top of it would be meaningless. Check the flow-map load, the "
    f"time-convention flip in FluxFlowMap._velocity, and GUIDANCE."
)

# --- gate 2: denoise(x, t) and map(x, t, 1) are genuinely different -------------------------------
# If they coincide, the two-timestep conditioning is inert and the load is suspect. denoise is a
# conditional mean, so it must be BLURRIER (lower variance) than the ODE endpoint, which is a sample.
t_probe = 0.5
gen = torch.Generator(device=DEVICE).manual_seed(0)
eps = torch.randn(health.shape, generator=gen, device=DEVICE, dtype=health.dtype)
x_t = SCHEDULE.alpha(t_probe) * health + SCHEDULE.sigma(t_probe) * eps
mean_est = flow_map.denoise(x_t, t_probe)
sample_est = flow_map.map(x_t, t_probe, 1.0)
rel_gap = float((mean_est - sample_est).norm() / sample_est.norm())
print(f"\nat t={t_probe}:  denoise std {float(mean_est.std()):.4f}   map(.,t,1) std "
      f"{float(sample_est.std()):.4f}   relative gap {rel_gap:.3f}")
assert rel_gap > 1e-3, (
    "denoise(x, t) == map(x, t, 1): the two-timestep conditioning is inert. Either the DualTimeEmbedder "
    "is not receiving a timestep pair, or the LoRA load was a no-op."
)
assert float(mean_est.std()) < float(sample_est.std()), (
    "the Tweedie mean is not blurrier than the ODE endpoint -- denoise and map are likely swapped"
)
assert torch.allclose(flow_map.map(x_t, t_probe, t_probe), x_t), "X_{t,t} must be the identity"

fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for ax, img in zip(axes, decode(health)):
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.axis("off")
fig.suptitle(f"PHASE 1 — untilted draws from p   (std {S_health:.4f}, prompt {PROMPT!r})")
plt.savefig(os.path.join(RUN_DIR, f"phase1_health_{RUN_TAG}.png"), dpi=90, bbox_inches="tight")
plt.show()
note(f"PHASE 1 GATE PASSED  latent_std={S_health:.4f} (base FLUX {S_BASE_FLUX:.4f})  "
     f"denoise-vs-map gap={rel_gap:.4f}  s/row={(time.time() - t0_health) / (8 * N_STEPS):.3f}")
if STOP_AFTER == "health":
    print("STOP_AFTER=health -- stopping here.")

  step 1/16  t=0.0000->0.0625  det    unguided         ESS/M=1.00  resampled=False  uniq/M=1.00  f mean=nan
  step 2/16  t=0.0625->0.1250  det    unguided         ESS/M=1.00  resampled=False  uniq/M=1.00  f mean=nan
  step 3/16  t=0.1250->0.1875  stoch  unguided         ESS/M=1.00  resampled=False  uniq/M=1.00  f mean=nan
  step 4/16  t=0.1875->0.2500  stoch  unguided         ESS/M=1.00  resampled=False  uniq/M=1.00  f mean=nan
  step 5/16  t=0.2500->0.3125  stoch  unguided         ESS/M=1.00  resampled=False  uniq/M=1.00  f mean=nan
  step 6/16  t=0.3125->0.3750  stoch  unguided         ESS/M=1.00  resampled=False  uniq/M=1.00  f mean=nan
  step 7/16  t=0.3750->0.4375  stoch  unguided         ESS/M=1.00  resampled=False  uniq/M=1.00  f mean=nan
  step 8/16  t=0.4375->0.5000  stoch  unguided         ESS/M=1.00  resampled=False  uniq/M=1.00  f mean=nan
  step 9/16  t=0.5000->0.5625  stoch  unguided         ESS/M=1.00  resampled=False  uniq/M=1.00  f mean=nan
  step 10/16  t=0.5625->0.62

### Phase 1 conclusions — gate passed (job 700463, L40S, 27 min)

| check | result | verdict |
|---|---|---|
| LoRA load | 692 tensors, 346 LoRA sites, $\max\lvert\Delta\rvert = 1.41$ on a fixed input | **not a no-op** — this is the flow map, not plain FLUX.1-dev |
| latent std | **1.0345** vs base FLUX's known-good 1.0150 | $p$ is not degraded |
| `denoise` vs `map(·,t,1)` at $t=0.5$ | std 0.9611 vs 0.9961, relative gap 0.166 | genuinely distinct, and the conditional **mean is blurrier than the sample** — exactly the ordering that would break if the two were conflated |
| $X_{t,t}$ | identity to float precision | flow-map algebra is right |
| decoded images | 8 clean, on-prompt dogs | the two-timestep conditioning produces real samples |
| throughput | **0.208 s/row** | 17% *faster* than the A6000's 0.251 the plan budgeted with |

**Three things this run changed downstream.**

1. **`S_ROW` 0.251 → 0.208.** All cost projections and the per-$m$ deadline check now use the measured
   L40S number. At this speed all of $m \in \{1,2,3\}$ fits in a 9 h job, so **`K` stays at the plan's
   operating point of 4** — no need to trade lookahead quality for wall clock.
2. **Setup costs ~22 min of every job**, essentially all of it loading ~34 GB of FLUX weights off the
   shared NFS volume (measured: 4% CPU, 51 s of CPU time in 20 min — I/O-bound, not compute). Budget it.
3. **`FluxFlowMap` now chunks its forward pass** (`max_rows=24`). The gate ran at $B=8$ and never
   exposed this, but reference generation calls `map` on $R = 64$ rows and the lookahead on $M K = 32$,
   against a transformer already holding ~24 GB. Chunking is *mathematically* identical but agrees only
   to float epsilon (measured $2.5\times10^{-7}$ relative), because BLAS picks different kernels per
   batch size — so `max_rows` is fixed for the run, like the denoiser cap, or $f$ would stop being a
   deterministic function of $x$.

**Not verified here, by design:** that $\text{denoise}$ is the *correct* Tweedie mean rather than merely
a different, blurrier function. Nothing cheap distinguishes those. What backs it is the algebra
(`test_flowmap_smc.py::test_score_at_matches_the_closed_form_and_tweedie`, exact to float64 on an
analytic model) plus the $\sigma$-ordering above. The reward's own sanity check in Phase 3 —
$f(\text{refs}) \approx (R-1)/R$ — is the next real constraint on the score field.

In [5]:
from creativity_measure import SquaredGlobalIEMDistance
from creativity_measure.tilt import NormalizedExpectedDistanceReward

# --- PHASE 3: the frozen reward --------------------------------------------------------------------
# These constants DEFINE f, so changing any of them changes the target. Seeds are load-bearing:
# distance seed 123, reference seed 7 (both carried over from the validated run 695271).
R, N_GAMMA, NUM_EPS = 64, 30, 3
REF_SEED, DIST_SEED, HELDOUT_SEED = 7, 123, 8
N_HELDOUT = 32

REFS_PATH = os.path.join(RUN_DIR, f"refs_R{R}_seed{REF_SEED}.pt")
LAMS_PATH = os.path.join(RUN_DIR, f"lam_s_R{R}_seed{REF_SEED}.json")


def _assert_same_p(saved: dict) -> None:
    """The reference file is keyed to the base process -- but NOT to `guid_window`.

    References must be generated by the same stepping the run uses, so a mismatch means f describes a
    different p than the one being sampled. `guid_window` is deliberately absent: it may vary freely
    across runs sharing one reference set, which is what makes a window sweep a fair comparison.
    """
    keyed = ("schedule", "n_steps", "ts", "stoch_window", "inside_step", "outside_step",
             "img_shape", "model_id", "prompt", "guidance", "max_denoiser_rows",
             "max_flow_map_rows")
    for k in keyed:
        got, want = saved["base_process"].get(k), BASE_PROCESS[k]
        if isinstance(want, tuple):
            got = tuple(got) if got is not None else got
        assert got == want, f"reference file was built under {k}={got!r}, this run uses {want!r}"
    assert "guid_window" not in saved["base_process"], "guid_window must not be keyed into p"


if DO_REFS:
    if os.path.exists(REFS_PATH):
        saved = torch.load(REFS_PATH, map_location="cpu", weights_only=False)
        _assert_same_p(saved)
        latent_refs = saved["x_refs"].to(DEVICE)
        heldout = saved["heldout"].to(DEVICE)
        print(f"references LOADED from {os.path.basename(REFS_PATH)} "
              f"(built on {saved.get('gpu', '?')}) -- the model is out of the reproducibility chain")
    else:
        # WHY THIS IS PERSISTED. Job 697271 aborted because references regenerated on a different GPU
        # shifted f by 16% of std_p(f) -- enough to change weight ratios by 1.25x. Saving the LATENTS
        # takes the model out of the chain, so every future leg gets a bit-identical reference set.
        t0 = time.time()
        latent_refs = draw_from_p(R, seed=REF_SEED)
        heldout = draw_from_p(N_HELDOUT, seed=HELDOUT_SEED)
        torch.save({"x_refs": latent_refs.detach().cpu(), "heldout": heldout.detach().cpu(),
                    "R": R, "seed": REF_SEED, "heldout_seed": HELDOUT_SEED,
                    "latent_dim": LATENT_DIM, "base_process": BASE_PROCESS,
                    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"},
                   REFS_PATH)
        print(f"references GENERATED in {(time.time() - t0) / 60:.1f} min and saved to "
              f"{os.path.basename(REFS_PATH)} -- future legs will load these")

    S = latent_refs.std().item()
    GAMMA_LO = max(1.0 / S**2, 1.0 / SIGMA_MAX**2)
    GAMMA_HI = min(2.0**10, 1.0 / SIGMA_MIN**2)
    GAMMAS = torch.logspace(math.log2(GAMMA_LO), math.log2(GAMMA_HI), N_GAMMA, base=2).to(DEVICE)

    D2 = SquaredGlobalIEMDistance(None, GAMMAS, num_eps=NUM_EPS, seed=DIST_SEED, score_fn=score_fn)
    t0 = time.time()
    reward = NormalizedExpectedDistanceReward(distance=D2, x_refs=latent_refs)   # builds the ref bank
    t_bank = time.time() - t0
    note(f"reward built: refs {tuple(latent_refs.shape)}  S={S:.4f}  gamma=[{GAMMA_LO:.4g}, {GAMMA_HI:.3g}] "
          f"x {N_GAMMA}   ref bank {t_bank / 60:.1f} min")

    # --- lambda_s: measured once, then CACHED --------------------------------------------------------
    # The two checks below cost (N_GAMMA-1)*NUM_EPS*(R + N_HELDOUT) = 8352 rows ~ 35 min. They are a
    # property of the (frozen) reward, so the run job loads the cached scalars instead of paying again.
    # `_assert_same_p` on the cache is what stops a stale lambda_s from silently calibrating a
    # different p. Delete lam_s_*.json to force a re-measure.
    cached = None
    if os.path.exists(LAMS_PATH):
        with open(LAMS_PATH) as fh:
            cached = json.load(fh)
        _assert_same_p(cached)
        assert cached["R"] == R and cached["n_gamma"] == N_GAMMA and cached["num_eps"] == NUM_EPS
        assert cached["dist_seed"] == DIST_SEED

    # A cache built on other hardware is NOT automatically trustworthy (see the "gpu" note where it is
    # written). Verify on a small subset rather than assume: 8 held-out samples = 696 rows, ~3 min.
    if cached is not None and cached.get("gpu") not in (None, GPU_NAME):
        vals = cached.get("f_heldout_values")
        if vals is None:
            print(f"\ncache was built on {cached.get('gpu')!r}, this job is on {GPU_NAME!r}, and it "
                  f"predates per-sample logging -- re-measuring lambda_s from scratch.")
            cached = None
        else:
            n_chk = min(8, len(vals))
            f_chk = reward(heldout[:n_chk])
            want = torch.tensor(vals[:n_chk], device=f_chk.device, dtype=f_chk.dtype)
            drift = float((f_chk - want).abs().max()) / cached["f_heldout_std"]
            note(f"cross-GPU check ({cached.get('gpu')} -> {GPU_NAME}): max|df| = "
                 f"{drift:.3f} x std_p(f) over {n_chk} held-out samples")
            if drift > 0.5:
                raise RuntimeError(
                    f"f drifted {drift:.2f}x std_p(f) across GPUs -- lambda_s = {cached['lam_s']:.2f} "
                    f"describes a different reward than this job computes. Delete "
                    f"{os.path.basename(LAMS_PATH)} to re-measure, or pin --constraint to "
                    f"{cached.get('gpu')!r}.")
            if drift > 0.15:
                print(f"  WARNING: drift is material ({drift:.2f}x std_p(f)); lambda_s is approximate.")

    if cached is not None:
        lam_s = cached["lam_s"]
        F_P_MEAN, F_P_STD = cached["f_heldout_mean"], cached["f_heldout_std"]
        note(f"lambda_s LOADED from {os.path.basename(LAMS_PATH)}: {lam_s:.3f}  "
              f"(E_p[f] = {F_P_MEAN:.4f} +- {F_P_STD:.4f}, f(refs) = {cached['f_refs_mean']:.4f})")
    else:
        # --- reward sanity: f on its OWN references centres near (R-1)/R, not 1 ---------------------
        # The numerator averages over all R refs INCLUDING the self-pair zero; the denominator is the
        # off-diagonal pair mean, which drops it. A value near 1.0 would mean the self-pair is being
        # excluded somewhere, i.e. f is not the function the calibration assumes.
        t0 = time.time()
        f_refs = reward(latent_refs)
        print(f"\nf(refs): mean {float(f_refs.mean()):.4f}  (expected ~(R-1)/R = {(R - 1) / R:.4f})   "
              f"std {float(f_refs.std()):.4f}   [{(time.time() - t0) / 60:.1f} min]")
        assert abs(float(f_refs.mean()) - (R - 1) / R) < 0.10, \
            "f on its own refs is off the expected centre"

        # --- lambda_s on a HELD-OUT batch, never on the references themselves -----------------------
        t0 = time.time()
        f_heldout = reward(heldout)
        F_P_MEAN, F_P_STD = float(f_heldout.mean()), float(f_heldout.std())
        lam_s = 1.0 / F_P_STD
        print(f"held-out f: mean {F_P_MEAN:.4f}  std {F_P_STD:.4f}  "
              f"({N_HELDOUT} samples, {(time.time() - t0) / 60:.1f} min)")
        note(f"lambda_s = 1/std_p(f) = {lam_s:.3f}  (E_p[f] = {F_P_MEAN:.4f} +- {F_P_STD:.4f})")
        with open(LAMS_PATH, "w") as fh:
            json.dump({"lam_s": lam_s, "f_heldout_mean": F_P_MEAN, "f_heldout_std": F_P_STD,
                       "f_refs_mean": float(f_refs.mean()), "S": S,
                       "R": R, "n_gamma": N_GAMMA, "num_eps": NUM_EPS,
                       "dist_seed": DIST_SEED, "base_process": BASE_PROCESS,
                       # Recorded so a later job on DIFFERENT hardware can VERIFY rather than trust.
                       # f is produced by running the bf16 transformer, so a different card changes the
                       # score field even though the reference latents come byte-identical off disk.
                       # Job 697271 died exactly here: a different GPU shifted f by 16% of std_p(f).
                       "gpu": GPU_NAME,
                       "f_heldout_values": [float(v) for v in f_heldout]}, fh, indent=2,
                      default=str)
        print(f"saved -> {os.path.basename(LAMS_PATH)}")
if STOP_AFTER == "refs":
    print("\nSTOP_AFTER=refs -- stopping here.")

references LOADED from refs_R64_seed7.pt (built on NVIDIA L40S) -- the model is out of the reproducibility chain
[   22.0 min | 10.2G] reward built: refs (64, 65536)  S=1.0606  gamma=[0.889, 1.02e+03] x 30   ref bank 19.2 min
[   22.0 min | 10.2G] lambda_s LOADED from lam_s_R64_seed7.json: 140.093  (E_p[f] = 1.0062 +- 0.0071, f(refs) = 0.9844)


### Phase 3 conclusions — reward frozen (job 700483, L40S, 36 min)

| quantity | value | reading |
|---|---|---|
| references | $R=64$ at $d=65536$, seed 7; held-out 32 at seed 8 | drawn by the **same stepping** the run uses, then persisted — the model is now out of the reproducibility chain |
| $S$ (latent std of refs) | 1.0606 | vs 1.0345 on the 8-sample gate batch and 1.0150 for base FLUX; 64 samples is the better estimate |
| $\gamma$ grid | $[0.889,\ 1.02\times10^{3}]\times 30$ | inside the denoiser's validated range |
| $f(\text{refs})$ | **0.984375** | $= 63/64 = (R-1)/R$ **exactly** — see below |
| $E_p[f]$ (held-out) | $1.0062 \pm 0.0071$ | $f$ is normalized, so $\approx 1$ under $p$ is correct by construction |
| $\lambda_s = 1/\operatorname{std}_p(f)$ | **140.093** | so $m=3$ means $\lambda = 420.28$ and $\lambda\cdot\operatorname{std}_p(f) = 3$ |
| ref-bank throughput | 5568 rows in 19.1 min | **0.206 s/row** on the reward path |

**The $f(\text{refs}) = 63/64$ check is stronger than "approximately right" — it is an exact identity.**
$f(x_r)$ divides by the *off-diagonal* pair mean but its own numerator averages over all $R$ references
**including the self-pair** $D^2(x_r,x_r) = 0$, so $\frac{1}{R}\sum_r f(x_r) = \frac{R-1}{R}$ algebraically,
with no approximation. Landing on 0.984375 to every digit therefore confirms three things at once: the
self-pair distances are *exactly* zero (the score field is deterministic — the same point gives the same
score), the off-diagonal masking in `reference_pair_mean` is correct, and the normalization is wired the
way the $\lambda$ calibration assumes. A value near 1.0 instead would have meant the self-pair was being
excluded somewhere and every $\lambda$ below would be miscalibrated.

**$f$ is extremely concentrated under $p$**: a coefficient of variation of 0.7% ($1.0062 \pm 0.0071$).
That is why $\lambda_s$ is large in absolute terms (140, against 92.2 for the pCN run's base-FLUX $p$ —
not directly comparable, since $p$ differs). What matters is the product: at $m=3$, $\lambda f$ spans
about 3 across the cloud, which is the weight spread `logsumexp` was chosen to handle.

**Caveat worth carrying:** $\operatorname{std}_p(f)$ is estimated from **32** held-out samples, so
$\lambda_s$ itself carries roughly $\pm 13\%$ ($1/\sqrt{2(n-1)}$). The $m$-sweep spans a factor of 3, so
this does not change which $m$ gets selected, but do not read $\lambda_s$ as a precise constant.

**Cost measured here fixes Phase 4's one open parameter.** At 0.206 s/row (and 0.231 on the narrower
flow-map path) the two sanctioned guidance windows differ by one guided step and by whether the third
$\lambda$ survives:

| `guid_window` | guided steps | h per $m$ | all three $m$ in 9 h? |
|---|---|---|---|
| $(0.1, 1.0)$ | 14 | 2.34 – 2.62 | **no** — third dropped at the pessimistic rate |
| $(0.1, 0.9)$ | 13 | 2.17 – 2.44 | **yes**, at both rates |

So Phase 4 runs $(0.1, 0.9)$. Because `guid_window` is deliberately **not** part of $p$, this needs no
reference rebuild and a later run can widen it back and still compare fairly against these same
references — which is the entire reason it was kept out of the base process.

In [6]:
# --- THE K SWEEP AT FIXED LAMBDA --------------------------------------------------------------------
# One job per K, all submitted concurrently. K and M_TILT come from the environment so that one
# notebook + one .slurm serves every cell of the sweep and nothing about the run differs between them
# except the swept variable.
M = int(os.environ.get("M_PARTICLES", 8))    # particles
K = int(os.environ.get("K_LOOK", 4))         # lookahead draws per particle per guided step -- SWEPT
ETA = 1.5                                    # SNR factor: g(t') = eta*g(t)

# ESS_THRESHOLD stays at 0.5 and is NOT a free parameter. It interpolates between two known endpoints:
# between resamples U_n = V_n - V_{last}, a long-baseline difference in which the intermediate V
# cancel; `_resample` zeroes U, forcing selection onto a SINGLE-STEP increment -- the noise-dominated
# quantity (sd(U)/sd(V) ~ 1.0). Threshold -> 0 is exact IS; threshold -> 1 selects on raw increments
# every step. At M = 8 the observed ESS/M sits in 0.14-0.9, so 0.7 and 0.8 are not a distinct regime:
# on a3_2 m=3 they fire on 15/16 and 16/16 steps against 1.0's 16/16. Both endpoints have been run
# (a3_1 at 1.0, a3_2 at 0.5) and E_q[f] falls under both.
ESS_THRESHOLD = 0.5

# WHERE THE REWARD RUNS. Full window: guidance on every step. (0.0, 1.0) is safe even though t = 0 is
# degenerate (g = inf, c = 0, gamma = 0): guidance is keyed on each step's TARGET time, which on a grid
# starting at 0 is never 0. The earliest reachable level is t = 0.0625, where t' = 0.0516, c = 0.057,
# gamma = 0.0030 and the renoise variance is 0.30 -- all well conditioned.
#
# The lambda = 0 CONTROL sets GUID_WINDOW=(1.0,1.0) instead, so only the mandatory terminal update
# fires and the job costs 16*M projection rows rather than 15*M*K lookahead rows. It is the untilted
# reference curve and the in-job E_p[f]. In-job matters: a GPU change once shifted f by 16% of
# std_p(f) (job 697271), the same order as the effects being measured here.
GUID_WINDOW = tuple(float(x) for x in os.environ.get("GUID_WINDOW", "0.0,1.0").split(","))

# ONE lambda, not a sweep. At M = 8 a lambda sweep measures degeneracy rather than the estimator:
# a3_2 collapses to a single root ancestor by step 2 at m = 2. m = 1 is too weak (the mechanism fires
# once in 16 steps); m = 1.25 is the conservative choice in the gap. The K=4 job is the live scout --
# if its first ~5 steps collapse, lambda is revised and EVERY cell re-runs at the new value.
M_LIST = [float(x) for x in os.environ.get("M_LIST", "1.25").split(",")]
SEED = 101

# Recording flags. Both consume no randomness and never touch U, so the trajectory is bit-identical to
# a run without them (tests/test_flowmap_smc.py::test_recording_flags_leave_the_run_bit_identical).
#   record_r_k       -- keeps (M, K) lookahead rewards, so every K' <= K is reconstructible offline
#                       by subsetting. One run at K answers for every smaller K'.
#   project_endpoint -- the honest per-step E_q[f]: push each particle to map(x, t, 1), evaluate f,
#                       and weight by softmax(U_pre) BEFORE resampling. `f_mean` is not a substitute --
#                       it is unweighted and taken at the renoised level t'.
RECORD_R_K = True
PROJECT_ENDPOINT = True


class DeadlineReached(RuntimeError):
    """Raised from the `on_step` callback when the wall clock runs out, to stop the SMC cleanly."""


def _hist(partial):
    """Per-step diagnostics, in the one shape used by both the checkpoint and the final save.

    Tensors are moved to CPU here rather than at the call sites: a preempted job's partial file is
    the only surviving record of the pre-collapse window, which is the deliverable, so it has to be
    loadable without a GPU.
    """
    def cpu_list(xs):
        return [None if v is None else v.detach().cpu() for v in xs]

    return {
        "ess_history": partial.ess_history, "resampled_history": partial.resampled_history,
        "uniq_history": partial.uniq_history, "t_history": partial.t_history,
        "t_prime_history": partial.t_prime_history, "guided_history": partial.guided_history,
        "f_mean": partial.f_mean, "f_std": partial.f_std,
        "f_min": partial.f_min, "f_max": partial.f_max,
        "l_mean": partial.l_mean, "l_std": partial.l_std,
        "s_mean": partial.s_mean, "s_std": partial.s_std,
        # --- the K-sweep additions --------------------------------------------------------------
        "eqf_history": partial.eqf_history,
        "V_history": cpu_list(partial.V_history),
        "U_pre_history": cpu_list(partial.U_pre_history),
        "resample_idx_history": cpu_list(partial.resample_idx_history),
        "r_k_history": cpu_list(partial.r_k_history),
        "f_proj_history": cpu_list(partial.f_proj_history),
    }


def make_checkpointer(path: str, config: dict):
    """Persist the cloud after every step, and stop the run at the deadline.

    The SMC is one long library call, so `on_step` is the only place a notebook can interrupt it.
    The callback is read-only and consumes no randomness, so the run is bit-identical to one without.
    Writes are temp-file + os.replace, so a kill mid-write cannot corrupt the last good file.

    THIS IS THE PREEMPTION INSURANCE. `killable` can take the job at any moment and Algorithm 3
    cannot resume, but the deliverable here is the PRE-COLLAPSE part of the curve, which is early --
    so a job killed at step 12 has already delivered what it was for.
    """
    def on_step(i, snap, partial):
        payload = {
            "step": i, "t": snap.t,
            "X": snap.X, "U": snap.U, "V": snap.V, "ancestors": snap.ancestors,
            "U_pre": snap.U_pre, "resample_idx": snap.resample_idx, "f_proj": snap.f_proj,
            **_hist(partial), "config": config, "partial": True,
        }
        tmp = f"{path}.tmp"
        torch.save(payload, tmp)
        os.replace(tmp, path)
        elapsed = (time.time() - config["t0"]) / 60
        eqf = partial.eqf_history[-1]
        note(f"  K={config['K']} m={config['m_tilt']:g} step {i + 1}/{N_STEPS} t={snap.t:.4f}  "
             f"ESS/M={partial.ess_history[-1] / config['M']:.2f}  "
             f"uniq/M={partial.uniq_history[-1]:.2f}  "
             f"E_q[f]={eqf:.4f}  f_cand={partial.f_mean[-1]:.4f}  "
             f"{elapsed:.1f} min into this m")
        if time.time() > DEADLINE:
            raise DeadlineReached(f"deadline after step {i + 1} (t={snap.t:.4f})")
    return on_step


# --- cost projection ---------------------------------------------------------------------------
# One reward call on B latents costs (N_GAMMA-1)*NUM_EPS*B denoiser rows. The lookahead runs at
# B = M*K; the projection adds B = M on every step (the terminal step reuses the reward it already
# computed, so it is free there).
if DO_RUN:
    _ts = [n / N_STEPS for n in range(N_STEPS + 1)]
    _guided = [n for n in range(N_STEPS - 1) if GUID_WINDOW[0] <= _ts[n + 1] <= GUID_WINDOW[1]]
    _rw = (N_GAMMA - 1) * NUM_EPS
    rew_rows = _rw * M * K
    net_rows = M * (2 * K + 2)
    t_guided = (rew_rows + net_rows) * S_ROW
    t_terminal = (_rw * M + M) * S_ROW
    t_unguided = M * S_ROW
    t_proj = (_rw * M + M) * S_ROW * (N_STEPS - 1) if PROJECT_ENDPOINT else 0.0
    t_run = (len(_guided) * t_guided + t_terminal
             + (N_STEPS - 1 - len(_guided)) * t_unguided + t_proj)
    budget_s = DEADLINE - time.time()
    print(f"M={M} K={K} guid_window={GUID_WINDOW}  ->  {len(_guided)} guided steps "
          f"x {t_guided / 60:.1f} min + terminal {t_terminal / 60:.1f} min "
          f"+ projection {t_proj / 60:.1f} min  =  {t_run / 3600:.2f} h")
    print(f"  reward is {rew_rows / max(rew_rows + net_rows, 1):.1%} of a guided step "
          f"({rew_rows} reward rows vs {net_rows} flow-map rows at {S_ROW} s/row)")
    print(f"budget left {budget_s / 3600:.2f} h -> {budget_s / t_run:.1f} of the "
          f"{len(M_LIST)} requested m values ({M_LIST})")

results, run_paths = {}, {}
if DO_RUN:
    for m_tilt in M_LIST:
        lam = m_tilt * lam_s
        tag = f"N{M}_K{K}_m{m_tilt:g}_seed{SEED}_{RUN_TAG}"
        path = os.path.join(RUN_DIR, f"partial_{tag}.pt")
        config = {
            "M": M, "K": K, "eta": ETA, "n_steps": N_STEPS, "lam": lam, "m_tilt": m_tilt,
            "lam_s": lam_s, "seed": SEED, "ess_threshold": ESS_THRESHOLD,
            "guid_window": GUID_WINDOW, "base_process": BASE_PROCESS,
            "R": R, "n_gamma": N_GAMMA, "num_eps": NUM_EPS, "dist_seed": DIST_SEED,
            "refs_file": os.path.basename(REFS_PATH), "t0": time.time(),
            # PINNED SO THE SWEEP CANNOT BE COMPARED ACROSS HARDWARE BY ACCIDENT. Cells run
            # concurrently on whatever frees up first; a GPU change shifted f by 16% of std_p(f)
            # once (job 697271), which is the same order as the effects here. The analysis asserts
            # every cell agrees on this string.
            "gpu_name": GPU_NAME, "run_tag": RUN_TAG,
            "record_r_k": RECORD_R_K, "project_endpoint": PROJECT_ENDPOINT,
        }
        if time.time() + t_run > DEADLINE:
            print(f"\nskipping m={m_tilt:g}: {t_run / 3600:.2f} h projected does not fit in the "
                  f"{(DEADLINE - time.time()) / 3600:.2f} h left")
            break
        note(f"=== K={K}  m={m_tilt:g}  lambda={lam:.2f}  (lambda_s={lam_s:.2f})  "
             f"M={M} N={N_STEPS} eta={ETA} guid_window={GUID_WINDOW} "
             f"ess_thr={ESS_THRESHOLD} gpu={GPU_NAME} ===")
        t0, timed_out = time.time(), False
        try:
            res = flowmap_smc_sample(
                reward, lam, M, **BASE_KW, mc_samples=K, eta=ETA, guid_window=GUID_WINDOW,
                ess_threshold=ESS_THRESHOLD, antithetic=True, keep_steps=True,
                record_r_k=RECORD_R_K, project_endpoint=PROJECT_ENDPOINT,
                on_step=make_checkpointer(path, config), seed=SEED, verbose=True,
            )
        except DeadlineReached as exc:
            timed_out = True
            note(f"*** STOPPED ON DEADLINE: {exc}")
            res = None
        mins = (time.time() - t0) / 60
        if res is not None:
            final = os.path.join(RUN_DIR, f"result_{tag}.pt")
            torch.save({"X": res.X.cpu(), "logw": res.logw.cpu(),
                        "steps": [{"t": s.t, "X": s.X, "U": s.U, "V": s.V,
                                   "ancestors": s.ancestors, "U_pre": s.U_pre,
                                   "resample_idx": s.resample_idx, "f_proj": s.f_proj}
                                  for s in (res.steps or [])],
                        **_hist(res),
                        "config": {**config, "wall_clock_min": mins}, "partial": False}, final)
            results[m_tilt], run_paths[m_tilt] = res, final
            note(f"K={K} m={m_tilt:g} COMPLETED in {mins:.1f} min ({mins / 60:.2f} h)  "
                 f"vs projected {t_run / 3600:.2f} h   saved -> {os.path.basename(final)}")
            # The headline: the within-run curve, and the give-back that summarises it.
            eqf = [v for v in res.eqf_history if v == v]
            if eqf:
                pk = max(eqf)
                note(f"K={K} m={m_tilt:g} SUMMARY  E_q[f] peak={pk:.4f} at step "
                     f"{res.eqf_history.index(pk) + 1}  terminal={res.eqf_history[-1]:.4f}  "
                     f"give-back={pk - res.eqf_history[-1]:+.4f}  "
                     f"final uniq/M={res.uniq_history[-1]:.3f}  "
                     f"resamples={sum(res.resampled_history)}/{N_STEPS}")
        else:
            print(f"m={m_tilt:g} incomplete after {mins:.1f} min; partial state in "
                  f"{os.path.basename(path)}")
            break

M=8 K=32 guid_window=(0.0, 1.0)  ->  15 guided steps x 78.3 min + terminal 2.4 min + projection 36.3 min  =  20.21 h
  reward is 97.7% of a guided step (22272 reward rows vs 528 flow-map rows at 0.206 s/row)
budget left 20.62 h -> 1.0 of the 1 requested m values ([1.25])
[   22.0 min | 10.2G] === K=32  m=1.25  lambda=175.12  (lambda_s=140.09)  M=8 N=16 eta=1.5 guid_window=(0.0, 1.0) ess_thr=0.5 gpu=NVIDIA L40S ===
[  103.1 min | 10.2G]   K=32 m=1.25 step 1/16 t=0.0625  ESS/M=0.84  uniq/M=1.00  E_q[f]=0.9172  f_cand=0.9184  81.0 min into this m
  step 1/16  t=0.0000->0.0625  det    guided tp=0.0516  ESS/M=0.84  resampled=False  uniq/M=1.00  f mean=0.9184
[  184.0 min | 10.2G]   K=32 m=1.25 step 2/16 t=0.1250  ESS/M=0.28  uniq/M=0.50  E_q[f]=0.9289  f_cand=0.9211  162.0 min into this m
  step 2/16  t=0.0625->0.1250  det    guided tp=0.1045  ESS/M=0.28  resampled=True  uniq/M=0.50  f mean=0.9211
[  265.2 min | 10.2G]   K=32 m=1.25 step 3/16 t=0.1875  ESS/M=0.36  uniq/M=0.12  E_q[f]=0.9621

In [7]:
# --- DIAGNOSTICS: this cell's own K, plus a cross-K overlay of whatever has landed -------------------
# Each job runs ONE K, so the within-job plot is a single curve. The overlay globs every result_*.pt
# AND partial_*.pt in RUN_DIR, so re-running this cell after later jobs finish builds Figure 1
# incrementally without re-running anything on the GPU.


def _collapse_step(uniq, m):
    """First step index (1-based) at which only one root ancestor survives, or None."""
    for i, u in enumerate(uniq):
        if u <= 1.0 / m + 1e-9:
            return i + 1
    return None


def _summarise(d):
    """Headline numbers for one run, restricted to what the pre-collapse window supports."""
    cfg, eqf, uniq = d["config"], d["eqf_history"], d["uniq_history"]
    coll = _collapse_step(uniq, cfg["M"])
    valid = [v for v in eqf if v == v]
    pk = max(valid) if valid else float("nan")
    # Everything right of the collapse step is one lineage's descendants, not an ensemble.
    pre = eqf[: (coll - 1)] if coll and coll > 1 else eqf
    pre_valid = [v for v in pre if v == v]
    return {
        "K": cfg["K"], "M": cfg["M"], "lam": cfg["lam"], "m_tilt": cfg["m_tilt"],
        "gpu_name": cfg.get("gpu_name", "?"), "partial": d.get("partial", False),
        "steps_done": len(eqf), "collapse_step": coll,
        "resamples": int(sum(d["resampled_history"])),
        "peak": pk, "peak_step": (eqf.index(pk) + 1) if valid else None,
        "terminal": eqf[-1] if eqf else float("nan"),
        "give_back": (pk - eqf[-1]) if valid else float("nan"),
        "pre_collapse_peak": max(pre_valid) if pre_valid else float("nan"),
        "final_uniq": uniq[-1] if uniq else float("nan"),
        "min_ess_over_M": min(d["ess_history"]) / cfg["M"] if d["ess_history"] else float("nan"),
    }


if DO_RUN:
    import glob

    rows = []
    for m_tilt, path in sorted(run_paths.items()):
        s = _summarise(torch.load(path, map_location="cpu", weights_only=False))
        rows.append(s)
        note(f"SUMMARY K={s['K']} m={s['m_tilt']:g} lam={s['lam']:.1f}  "
             f"peak={s['peak']:.4f}@{s['peak_step']}  terminal={s['terminal']:.4f}  "
             f"give-back={s['give_back']:+.4f}  collapse@{s['collapse_step']}  "
             f"resamples={s['resamples']}  uniq/M={s['final_uniq']:.3f}")

    # --- cross-K overlay: every result (and partial) currently on disk -----------------------------
    # Partials are included deliberately. `killable` can preempt at any moment and Algorithm 3 cannot
    # resume, but the deliverable is the PRE-COLLAPSE part of the curve, which is early -- so a job
    # killed at step 12 has already delivered what it was for.
    finals = sorted(glob.glob(os.path.join(RUN_DIR, "result_*.pt")))
    partials = sorted(glob.glob(os.path.join(RUN_DIR, "partial_*.pt")))
    curves, done = [], set()
    for fp in finals + partials:
        d = torch.load(fp, map_location="cpu", weights_only=False)
        c = d["config"]
        key = (c["K"], c["m_tilt"], c["M"])
        if key in done:                       # a finished result supersedes its own partial
            continue
        done.add(key)
        curves.append((c, d))

    gpus = {c.get("gpu_name", "?") for c, _ in curves}
    if len(gpus) > 1:
        note(f"*** WARNING: results span MORE THAN ONE GPU MODEL {gpus} -- a GPU change shifted f "
             f"by 16% of std_p(f) once (job 697271), the same order as the effects here. "
             f"The K comparison is NOT valid across these.")

    fig, ax = plt.subplots(figsize=(9, 5.5))
    for c, d in sorted(curves, key=lambda cd: (cd[0]["m_tilt"], cd[0]["K"])):
        eqf, uniq = d["eqf_history"], d["uniq_history"]
        xs = list(range(1, len(eqf) + 1))
        untilted = c["m_tilt"] == 0.0
        lab = (f"lambda=0 control (M={c['M']})" if untilted
               else f"K={c['K']}  m={c['m_tilt']:g}" + ("  [partial]" if d.get("partial") else ""))
        (line,) = ax.plot(xs, eqf, marker="o", ms=3.5, lw=1.6,
                          ls="--" if untilted else "-",
                          color="0.45" if untilted else None, label=lab)
        for i, r in enumerate(d["resampled_history"][: len(eqf)]):
            if r:
                ax.plot(i + 1, eqf[i], marker="x", ms=7, color=line.get_color())
        coll = _collapse_step(uniq, c["M"])
        if coll and not untilted:
            ax.axvline(coll, color=line.get_color(), ls=":", lw=1.1, alpha=0.7)

    ax.axhline(F_P_MEAN, color="k", lw=1.0)
    ax.axhspan(F_P_MEAN - F_P_STD, F_P_MEAN + F_P_STD, color="k", alpha=0.07)
    ax.text(0.4, F_P_MEAN, " E_p[f] +- std_p(f) (cached)", va="bottom", fontsize=8)
    ax.set_xlabel("step n   (t = n/16)")
    ax.set_ylabel("$E_q[f]$  (softmax($U_{pre}$)-weighted, pre-resample)")
    ax.set_title("Algorithm 3: within-run $E_q[f]$ by lookahead depth $K$ at fixed $\\lambda$\n"
                 "x = resampled   dotted vertical = collapse to one lineage "
                 "(nothing right of it is evidence)")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    out_png = os.path.join(RUN_DIR, f"eqf_by_K_{RUN_TAG}.png")
    fig.savefig(out_png, dpi=140)
    note(f"wrote {os.path.basename(out_png)} with {len(curves)} curve(s)")

    with open(os.path.join(RUN_DIR, f"summary_{RUN_TAG}.json"), "w") as fh:
        json.dump(rows, fh, indent=2, default=str)
    plt.show()

[ 1239.4 min | 10.2G] SUMMARY K=32 m=1.25 lam=175.1  peak=1.0479@10  terminal=1.0030  give-back=+0.0449  collapse@3  resamples=4  uniq/M=0.125
[ 1239.4 min | 10.2G] wrote eqf_by_K_k32.png with 5 curve(s)
